# Qwen3.5-4B: math problem → Lean 4 program

Loads `Qwen/Qwen3.5-4B`, sends a system prompt and a question through the chat template, and prints **only the Lean 4 program** from the response. The model reasons in a `<think>` block with a fixed token budget; if it runs out, thinking is cut off and the model is forced to write the program anyway, so you always get Lean code back. The reasoning is hidden by default. The program is also saved to `solution.lean`.

**Kaggle settings:** Accelerator = GPU (T4 x2 or P100), Internet = On (needed for the pip upgrade and the HF download).
If you attached the model as a Kaggle Model / Dataset instead, set `MODEL_PATH` below to its folder under `/kaggle/input/...`.

In [ ]:
# Qwen3.5 needs a recent transformers release (Kaggle's preinstalled one is usually too old).
!pip install -q -U "transformers>=5.2" accelerate

In [ ]:
import re
import time
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

print("torch", torch.__version__, "| transformers", transformers.__version__)
print("CUDA:", torch.cuda.is_available(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# ---- Config ----
MODEL_PATH = "Qwen/Qwen3.5-4B"   # or e.g. "/kaggle/input/qwen3.5/transformers/4b/1"

SYSTEM_PROMPT = """You are an elite mathematical problem solver with International Mathematical Olympiad (IMO) level expertise. Your goal is to find the correct answer to the given problem and to express your solution as a single Lean 4 program.

Work through the problem privately before writing any output:
1. UNDERSTAND: Carefully read and restate the problem. Identify what is given, what must be found, and all constraints.
2. EXPLORE: Consider multiple solution strategies. Think about relevant theorems, techniques, patterns, and analogous problems. Do not commit to one approach immediately.
3. PLAN: Select the most promising approach and outline the key steps before executing.
4. EXECUTE: Work through the solution methodically.
5. VERIFY: Check the answer by substituting back, testing edge cases, or using an alternative method. Ensure logical consistency throughout.

Be efficient: do not repeat work you have already done. Once the answer is verified, stop thinking and write the program.

Output format (strict):
- Your final output must be ONLY a Lean 4 program in a single ```lean4 code block. Write no text before or after the code block.
- The program must be complete and compile with Lean 4 and Mathlib: start with `import Mathlib`.
- State the final answer as a definition, e.g. `def answer : ℕ := ...`, and a `theorem` that formalizes the problem with that answer and proves it.
- Keep `--` comments short; do not restate your reasoning in the program.
- Do not use `sorry`, `admit`, or new `axiom`s."""

QUESTION_PROMPT = """<<PUT YOUR QUESTION HERE>>"""

ENABLE_THINKING = True   # False skips reasoning entirely (fastest, but usually less accurate)
THINK_BUDGET = 6144      # max reasoning tokens; past this, thinking is cut off and the model must answer
ANSWER_BUDGET = 2048     # max tokens for the Lean program itself
SHOW_THINKING = False    # set True to also print the model's reasoning
# Qwen advises against greedy decoding in thinking mode: it tends to loop and repeat
# itself until it hits the token limit. These are Qwen's recommended sampling settings.
DO_SAMPLE = True
TEMPERATURE, TOP_P, TOP_K = 0.6, 0.95, 20
OUTPUT_FILE = "solution.lean"

In [ ]:
# T4 / P100 have no bf16 support, so fall back to fp16 there.
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

t0 = time.time()
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, dtype=dtype, device_map="auto")
except (ValueError, KeyError) as e:
    # Qwen3.5 checkpoints are vision-language; if the causal-LM mapping is missing,
    # load the full image-text-to-text model and use it for text only.
    print("AutoModelForCausalLM failed, falling back:", e)
    from transformers import AutoModelForImageTextToText
    model = AutoModelForImageTextToText.from_pretrained(MODEL_PATH, dtype=dtype, device_map="auto")
model.eval()
print(f"Loaded {type(model).__name__} in {time.time() - t0:.1f}s")

In [ ]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": QUESTION_PROMPT},
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=ENABLE_THINKING,
)
prompt_ids = tokenizer(text, return_tensors="pt")["input_ids"].to(model.device)
print(f"Prompt tokens: {prompt_ids.shape[1]}")

sample_kwargs = dict(do_sample=DO_SAMPLE)
if DO_SAMPLE:
    sample_kwargs.update(temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K)

def encode(s):
    return tokenizer(s, add_special_tokens=False, return_tensors="pt")["input_ids"].to(model.device)

# The answer is pre-filled with the opening of the code block, so the model goes straight to Lean.
LEAN_PREFILL = "```lean4\nimport Mathlib\n"
# Qwen's suggested wording for cutting reasoning short once the budget is used up.
FORCE_STOP = "\n\nConsidering the limited time, I have to give the solution based on the thinking directly now.\n</think>\n\n"

think_end_id = tokenizer.convert_tokens_to_ids("</think>")
eos_ids = model.generation_config.eos_token_id
eos_ids = [eos_ids] if isinstance(eos_ids, int) else list(eos_ids or [])

t0 = time.time()
with torch.inference_mode():
    # Phase 1: reasoning, stopped at </think> or after THINK_BUDGET tokens.
    thinking = ""
    ids = prompt_ids
    if ENABLE_THINKING:
        out = model.generate(ids, max_new_tokens=THINK_BUDGET, eos_token_id=[think_end_id] + eos_ids, **sample_kwargs)
        think_ids = out[0, ids.shape[1]:]
        closed = len(think_ids) > 0 and think_ids[-1].item() == think_end_id
        while len(think_ids) > 0 and think_ids[-1].item() in eos_ids:  # model stopped without closing </think>
            think_ids = think_ids[:-1]
        thinking = tokenizer.decode(think_ids, skip_special_tokens=True).replace("<think>", "").strip()
        print(f"Thinking: {len(think_ids)} tokens ({'finished' if closed else 'CUT OFF at budget'})")
        ids = torch.cat([ids, think_ids.unsqueeze(0), encode("\n\n" if closed else FORCE_STOP)], dim=1)

    # Phase 2: the Lean program, stopped at the closing ``` fence.
    ids = torch.cat([ids, encode(LEAN_PREFILL)], dim=1)
    out = model.generate(ids, max_new_tokens=ANSWER_BUDGET, stop_strings=["```"], tokenizer=tokenizer, **sample_kwargs)
    answer_ids = out[0, ids.shape[1]:]

answer = tokenizer.decode(answer_ids, skip_special_tokens=True)
print(f"Program: {len(answer_ids)} tokens | total time {time.time() - t0:.1f}s")
if "```" not in answer:
    print("WARNING: hit ANSWER_BUDGET before the code block closed; the program is probably cut off.")

In [ ]:
if SHOW_THINKING and thinking:
    print("=" * 30, "THINKING", "=" * 30)
    print(thinking)
    print("=" * 70)

# The code block was opened by the prefill, so the program is everything up to the closing fence.
lean_program = ("import Mathlib\n" + answer.split("```", 1)[0]).strip()

if re.search(r"\b(sorry|admit)\b", lean_program):
    print("WARNING: the program contains `sorry`/`admit`.")

with open(OUTPUT_FILE, "w") as f:
    f.write(lean_program + "\n")

print(lean_program)

## Check the proof with LeanInteract

Compiles `solution.lean` against Mathlib using [LeanInteract](https://github.com/augustepoiroux/LeanInteract) (a Python wrapper around the Lean REPL).

The first run installs Lean (elan), creates a Lean project that depends on Mathlib and downloads Mathlib's prebuilt cache. That takes several minutes and ~5 GB of disk and needs Internet on. Later runs in the same session reuse it.

The proof is accepted only if Lean reports no errors, there are no `sorry`s, and every theorem depends only on Lean's standard axioms (`propext`, `Classical.choice`, `Quot.sound`).

In [ ]:
!pip install -q lean-interact
!install-lean   # installs elan (the Lean toolchain manager) into ~/.elan

import os
os.environ["PATH"] = os.path.expanduser("~/.elan/bin") + os.pathsep + os.environ["PATH"]
!lake --version

In [ ]:
from lean_interact import LeanREPLConfig, LeanServer, FileCommand, Command, TempRequireProject
from lean_interact.interface import LeanError

# Lean version for the checker; Mathlib is pinned to the matching tag.
# The model writes against whatever Mathlib it saw in training, so a mismatch can cause
# "unknown identifier" errors from renamed lemmas. Try another version if that happens.
LEAN_VERSION = "v4.19.0"
LEAN_TIMEOUT = 600  # seconds for one check (importing Mathlib alone takes a while)
STANDARD_AXIOMS = {"propext", "Classical.choice", "Quot.sound"}

t0 = time.time()
lean_config = LeanREPLConfig(project=TempRequireProject(lean_version=LEAN_VERSION, require="mathlib"), verbose=True)
lean_server = LeanServer(lean_config)
print(f"Lean {LEAN_VERSION} + Mathlib ready in {time.time() - t0:.1f}s")

In [ ]:
def check_lean_file(path):
    """Compile a Lean file and audit its axioms. Returns True if the proof is fully verified."""
    source = open(path).read()
    res = lean_server.run(FileCommand(path=os.path.abspath(path)), timeout=LEAN_TIMEOUT)
    if isinstance(res, LeanError):
        print("REPL ERROR:", res.message)
        return False

    for m in res.messages:
        if m.severity in ("error", "warning"):
            print(f"[{m.severity}] line {m.start_pos.line}, col {m.start_pos.column}: {m.data}")
    errors = res.get_errors()
    compiles = res.lean_code_is_valid(allow_sorry=False)

    # Audit axioms of every theorem/lemma: catches sorry (sorryAx), native_decide
    # (Lean.ofReduceBool) and any `axiom` the model declared to "prove" its claim.
    names = re.findall(r"^\s*(?:theorem|lemma)\s+([^\s:({\[]+)", source, re.MULTILINE)
    axioms_ok = True
    if compiles:
        if not names:
            print("No theorem/lemma found: nothing is actually proved.")
            axioms_ok = False
        for name in names:
            ax = lean_server.run(Command(cmd=f"#print axioms {name}", env=res.env), timeout=LEAN_TIMEOUT)
            text = " ".join(m.data for m in ax.messages) if not isinstance(ax, LeanError) else ax.message
            used = set(re.findall(r"[\w.]+", text.split("depends on axioms:", 1)[1])) if "depends on axioms:" in text else set()
            extra = used - STANDARD_AXIOMS
            print(f"{name}: {text.strip()}")
            if extra:
                print(f"  -> non-standard axioms: {sorted(extra)}")
                axioms_ok = False

    verified = compiles and axioms_ok
    print("=" * 70)
    print(f"errors: {len(errors)} | sorries: {len(res.sorries)} | theorems: {names}")
    print("VERIFIED ✅" if verified else "NOT VERIFIED ❌")
    return verified

t0 = time.time()
verified = check_lean_file(OUTPUT_FILE)
print(f"Checked in {time.time() - t0:.1f}s")